# Predictive Model for 12-Month Hospitalization Risk in Senior Living Populations

**Jessica Steele** | October 2025

## Executive Summary

This project develops a machine learning model to predict 12-month hospitalization risk for patients in senior living facilities using the CMS SynPUF (Synthetic Public Use Files) Sample 1 dataset. The final Gradient Boosting model achieves a ROC-AUC of 0.7558 and recall of 67.1% at the recommended threshold, enabling proactive identification of high-risk patients before adverse events occur.

> **Data Source Note:** This analysis uses the CMS DE-SynPUF synthetic dataset, which mimics real Medicare claims data while protecting patient privacy. Model performance estimates are based on synthetic data patterns and require validation on actual facility data before clinical deployment.

## 1. Problem Statement and Background

Unplanned hospitalizations in skilled nursing facility (SNF) and assisted living populations result in significant costs, care disruptions, and negative health outcomes. For organizations operating SNFs and overseeing assisted living residents, early identification of at-risk patients enables proactive care coordination, potentially reducing hospitalization rates while improving patient outcomes and satisfaction.

Hospitalizations from long-term care settings present many challenges. Care transitions between SNFs or assisted living facilities and acute care hospitals introduce risks of medication errors, infection exposure, and functional decline in elderly patients. From an organizational perspective, hospitalizations disrupt continuity of care, reduce census stability, and impact facility quality metrics. For SNF operators, hospital readmissions can affect Medicare reimbursement rates and quality star ratings.

**Project Objectives:**
- Identify patients at high risk of hospitalization within 12 months
- Provide risk stratification for facility-based care coordination teams
- Balance sensitivity with operational feasibility
- Support facility-level quality improvement and value-based care initiatives

## 2. Data and Features

### 2.1 Dataset Characteristics

The analysis utilized the CMS DE-SynPUF (Data Entrepreneurs' Synthetic Public Use Files) Sample 1 dataset, containing synthesized Medicare claims information designed to mimic actual beneficiary data while protecting patient privacy.

**Temporal Structure:**

```
[2008: Feature Calculation Period] → [Prediction Point: Jan 1, 2009] → [2009: Outcome Period]
```

The processed dataset contained **116,352 patients** with the following characteristics:

| Characteristic | Detail |
|---|---|
| Target Variable | Binary indicator of hospitalization during 2009 |
| Class Distribution | 16.2% positive cases (18,816 hospitalizations) |
| Feature Set | 36 features: demographics, chronic conditions, healthcare utilization |
| Baseline Period | Calendar year 2008 |

### 2.2 Exploratory Data Analysis

**Population Demographics**

The age distribution shows a predominantly elderly population centered around ages 65–75, with a right-skewed distribution extending to age 100, consistent with Medicare beneficiary populations in senior living settings.

**Chronic Disease Burden**

The distribution of chronic conditions is heavily right-skewed — approximately 45,000 patients had no documented chronic conditions, with frequency decreasing progressively for higher comorbidity counts. Patients with higher chronic condition counts show elevated hospitalization risk, increasing from approximately 5% for patients with no conditions to over 45% for patients with 10 or more conditions.

**Baseline Utilization Patterns**

Most patients (over 100,000) had zero baseline hospital admissions in 2008. Patients with prior admissions showed substantially higher future hospitalization risk (30–40%) compared to those without (approximately 15%).

![Exploratory analysis: (A) Age distribution, (B) Chronic condition count, (C) Class balance, (D) Baseline hospitalization history, (E) Risk by baseline utilization, (F) Risk by comorbidity burden](figures/eda_hospitalization_risk.png)

### 2.3 Feature Correlations

Correlation analysis identified moderate to strong relationships among utilization features:

- Hospital admissions, hospital stays, and inpatient costs showed strong intercorrelation (r = 0.79–0.83)
- Number of chronic conditions correlated moderately with baseline admissions (r = 0.51) and inpatient costs (r = 0.41)
- Age showed weak correlations with other features (r < 0.08)
- The target variable correlated most strongly with chronic conditions (r = 0.31), followed by baseline admissions (r = 0.15)

These patterns informed feature selection and suggested that both clinical complexity and prior utilization are predictors of future hospitalization risk.

![Correlation matrix for key features. Strong correlations (r > 0.75) among utilization metrics; target variable most correlated with chronic condition count (r = 0.31).](figures/correlation_matrix.png)

### 2.4 Feature Categories

The model incorporates **36 features** across three primary categories:

| Category | Features |
|---|---|
| **Demographics** | Age, Gender |
| **Clinical Characteristics** | Number of chronic conditions; individual condition indicators (Alzheimer's, CHF, CKD, cancer, COPD, depression, diabetes, ischemic heart disease, osteoporosis, arthritis, stroke); composite risk scores (cardiovascular burden, high-risk conditions count); disease category flags (respiratory, renal, cognitive impairment) |
| **Healthcare Utilization (2008)** | Hospital admissions, hospital days, inpatient costs, ED visits (est.), outpatient visits/costs, physician visits, total visits, cost per hospital day, total costs; utilization flags (prior utilizer, frequent user, high ER user, high outpatient user); composite scores (ER utilization, age-risk) |

## 3. Methodology

### 3.1 Model Development Process

- **Data Splitting**: Three-way stratified split maintaining strict separation between model development and evaluation:
  - Training set: 70% (n=81,445) — model training and hyperparameter tuning
  - Validation set: 10% (n=11,636) — threshold optimization only
  - Test set: 20% (n=23,271) — final evaluation only (touched once)
- **Cross-Validation**: 5-fold stratified CV on training set only
- **Hyperparameter Tuning**: Grid search with 3-fold CV on training set only
- **Threshold Optimization**: Systematic evaluation on validation set across operational ranges (0.10–0.50)
- **Final Evaluation**: Single evaluation on held-out test set with pre-selected threshold

> **Methodological Note:** The threshold was optimized exclusively on the validation set to prevent data leakage. The test set remained completely untouched until final evaluation, ensuring reported performance metrics represent true generalization to unseen data.

### 3.2 Cross-Validation Results

All models demonstrated stable performance across folds on the training set:

| Model | CV ROC-AUC | Std Dev | Validation ROC-AUC | Test ROC-AUC |
|---|---|---|---|---|
| Logistic Regression | 0.7512 | 0.0060 | 0.7437 | 0.7520 |
| Random Forest | 0.7518 | 0.0061 | 0.7493 | 0.7546 |
| Gradient Boosting | 0.7544 | 0.0059 | 0.7504 | 0.7558 |

The close alignment between CV, validation, and test performance (difference < 0.006) indicates good generalization without overfitting. Low standard deviations (< 0.007) demonstrate model stability.

### 3.3 Hyperparameter Optimization

**Gradient Boosting** (selected via grid search on training set):
- `n_estimators`: 100 &ensp;|&ensp; `learning_rate`: 0.05 &ensp;|&ensp; `max_depth`: 4

**Random Forest:**
- `n_estimators`: 200 &ensp;|&ensp; `max_depth`: 10 &ensp;|&ensp; `min_samples_split`: 10 &ensp;|&ensp; `min_samples_leaf`: 5

### 3.4 Model Selection

Three candidate algorithms — Logistic Regression, Random Forest, and Gradient Boosting — were trained with `class_weight='balanced'` to address class imbalance (16.2% positive). All tuning and selection decisions were based exclusively on training set cross-validation and validation set performance.

### 3.5 Class Imbalance Treatment

Initial models without class balancing exhibited high precision but critically low recall (2.6%), rendering them unsuitable for clinical application. Class weighting substantially improved performance by penalizing misclassification of the minority class.

**Performance Comparison (Gradient Boosting):**

| Configuration | Precision | Recall | F1 Score | ROC-AUC |
|---|---|---|---|---|
| Unbalanced | 0.550 | 0.026 | 0.050 | 0.756 |
| Balanced | 0.294 | 0.671 | 0.409 | 0.756 |

The balanced model increased recall by a factor of **25.8x** while maintaining comparable discrimination ability.

## 4. Results

### 4.1 Model Comparison (Validation Set)

| Model | ROC-AUC | PR-AUC | Precision | Recall | F1 Score |
|---|---|---|---|---|---|
| Logistic Regression (Balanced) | 0.744 | 0.331 | 0.276 | 0.692 | 0.394 |
| Random Forest (Balanced) | 0.749 | 0.341 | 0.265 | 0.768 | 0.394 |
| Gradient Boosting (Balanced) | 0.750 | 0.343 | 0.284 | 0.666 | 0.398 |

Gradient Boosting was selected based on highest ROC-AUC (0.750) and PR-AUC (0.343) with balanced precision-recall trade-off.

![ROC and Precision-Recall curves for all three balanced models on the validation set. Gradient Boosting (AUC=0.7504, PR-AUC=0.3434) slightly outperforms alternatives.](figures/roc_pr_curves.png)

### 4.2 Test Set Performance (Final Holdout Evaluation)

The selected Gradient Boosting model was evaluated once on the held-out test set at threshold 0.20:

| Metric | Value |
|---|---|
| ROC-AUC | 0.7558 |
| PR-AUC | 0.3440 |
| Precision | 0.2943 |
| Recall | 0.6710 |
| F1 Score | 0.4091 |
| Patients flagged | 8,587 (36.9%) |

**Performance Consistency Across Sets:**
- Training CV: 0.7544 ± 0.0059
- Validation: 0.7504
- Test: 0.7558

Minimal variation across all three sets (< 0.006) confirms strong generalization.

### 4.3 Feature Importance

The top five predictive features identified by the Gradient Boosting model:

| Rank | Feature | Importance |
|---|---|---|
| 1 | Number of chronic conditions | 0.60 |
| 2 | Total baseline visits | 0.08 |
| 3 | ER utilization score | 0.06 |
| 4 | High-risk conditions indicator | 0.05 |
| 5 | Baseline outpatient cost | 0.04 |

The dominance of chronic condition count (**60%** of total importance) suggests that disease burden is the primary driver of hospitalization risk, followed by healthcare utilization patterns.

![Top 20 feature importances for the Gradient Boosting model.](figures/feature_importance.png)

### 4.4 Risk Stratification and Calibration

The model demonstrates effective risk stratification and strong calibration across four tiers:

| Risk Level | Patient Count | Avg Predicted Probability | Actual Hospitalization Rate |
|---|---|---|---|
| Low Risk (<15%) | 55,866 | 5.7% | 5.4% |
| Moderate Risk (15–25%) | 33,018 | 19.8% | 19.7% |
| High Risk (25–35%) | 18,740 | 29.2% | 29.1% |
| Very High Risk (>35%) | 8,728 | 42.2% | 44.3% |

The close alignment between predicted probabilities and actual rates indicates good model calibration, supporting reliable risk stratification for clinical use.

## 5. Threshold Analysis

### 5.1 Threshold Selection Framework

The decision threshold was optimized on the validation set to prevent data leakage. Three configurations are presented based on organizational capacity:

| Approach | Threshold | Flagged % | Recall | Precision | F1 |
|---|---|---|---|---|---|
| **High-Recall** | 0.15 | ~52% | 83.6% | 25.5% | 0.390 |
| **Balanced (recommended)** | 0.20 | ~37% | 66.6% | 28.4% | 0.398 |
| **Precision-Focused** | 0.25 | ~24% | 48.5% | 32.6% | 0.390 |

### 5.2 Validation Set Threshold Analysis

| Threshold | Precision | Recall | F1 Score | Flagged % |
|---|---|---|---|---|
| 0.10 | 0.2331 | 0.9352 | 0.3731 | 64.9% |
| 0.15 | 0.2546 | 0.8359 | 0.3903 | 53.1% |
| 0.20 | 0.2840 | 0.6660 | 0.3982 | 37.9% |
| 0.25 | 0.3257 | 0.4849 | 0.3897 | 24.1% |
| 0.30 | 0.3804 | 0.3319 | 0.3545 | 14.1% |
| 0.35 | 0.4281 | 0.2023 | 0.2748 | 7.6% |
| 0.40 | 0.4600 | 0.1190 | 0.1890 | 4.2% |
| 0.45 | 0.4656 | 0.0611 | 0.1080 | 2.1% |
| 0.50 | 0.5130 | 0.0313 | 0.0591 | 1.0% |

The optimal threshold of **0.20** was selected based on maximizing F1 score on the validation set.

![Left: Precision, Recall, and F1 vs. threshold with optimal F1 at 0.20. Right: Percentage of population flagged at each threshold.](figures/threshold_analysis.png)

### 5.3 Test Set Confusion Matrix

At the recommended threshold of 0.20 on the test set (n=23,271):

|  | Predicted Negative | Predicted Positive |
|---|---|---|
| **Actual Negative** | 14,745 (TN) | 4,760 (FP) |
| **Actual Hospitalized** | 1,239 (FN) | 2,527 (TP) |

The contrast between thresholds 0.20 and 0.50 (which achieves only 2.5% recall) demonstrates why default thresholds are inappropriate for imbalanced datasets in clinical applications.

![Confusion matrices at threshold 0.20 (recall=0.671, precision=0.294) vs. default 0.50 (recall=0.025, precision=0.530), showing why threshold selection is critical for imbalanced clinical datasets.](figures/confusion_matrices.png)

### 5.4 Example Patient Predictions (Test Set)

| Case | Age | Chronic Conditions | Prior Admissions | Baseline Visits | Predicted Risk | Risk Level | Actual Outcome |
|---|---|---|---|---|---|---|---|
| 1 | 83 | 6 | 1 | Minimal | 33.9% | High | Not hospitalized |
| 2 | 69 | 1 | 0 | None | 9.2% | Low | Not hospitalized |
| 3 | 79 | 1 | 0 | 7 visits | 11.1% | Low | Not hospitalized |
| 4 | 73 | 3 | 0 | None | 19.2% | Moderate | Hospitalized |
| 5 | 50 | 0 | 0 | None | 2.7% | Low | Not hospitalized |

These cases illustrate how the model integrates age, chronic conditions, and utilization patterns to generate individualized risk scores.

## 6. Limitations

- **Synthetic Data**: The CMS DE-SynPUF data mimics real Medicare claims but may not fully capture actual patient patterns. Real-world validation is required before clinical deployment.
- **Class Imbalance**: Despite class weighting, the 5:1 negative-to-positive ratio limits achievable precision. Approximately 70% of flagged patients will not experience hospitalization within the prediction window.
- **Temporal Considerations**: The model predicts 12-month risk based on prior-year utilization. Performance may degrade if healthcare delivery patterns change, population characteristics shift, or interventions successfully reduce hospitalizations among high-risk patients. Regular retraining is recommended.
- **Feature Availability**: Deployment requires complete diagnosis history, a calendar year of utilization metrics, and current demographic information from electronic health records.

## 7. Conclusions

This project demonstrates that machine learning can identify patients at high risk of hospitalization in senior living populations. The final Gradient Boosting model with class balancing achieves ROC-AUC of 0.7558 and recall of 67.1% at the recommended threshold, representing a practical balance between case detection and operational feasibility.

**Key findings:**

- **Class balancing is essential**: Without class weighting, recall was only 2.6%. Balanced weighting increased recall to 67.1% while maintaining discrimination ability.
- **Threshold selection requires organizational context**: Options range from high-recall (83.6% recall, 52% flagged) to precision-focused (48.5% recall, 24% flagged).
- **Chronic disease burden is the primary predictor**: Number of chronic conditions accounts for 60% of model feature importance.
- **Model calibration supports clinical use**: Predicted probabilities align closely with actual hospitalization rates across risk categories.
- **Synthetic data requires real-world validation**: Performance estimates require confirmation on actual facility data before deployment.

## References

Centers for Medicare & Medicaid Services. (n.d.). *DE-SynPUF: CMS 2008–2010 Data Entrepreneurs' Synthetic Public Use File*. Retrieved from https://www.cms.gov/Research-Statistics-Data-and-Systems/Downloadable-Public-Use-Files/SynPUFs